In [ ]:
import base64
import hashlib
import os
from pathlib import Path
import subprocess
import sys
import tempfile
import zlib

RUNNER_SHA256 = 'ffd50bcdd871df18b1dd4ad91897247264e848d690a4544dca385cfbdc141866'
PAYLOAD_SHA256 = '395841edec72509cf8d6c48792391d5f8d61002794001e99fde4aebd6a9067d5'
RUNNER_B64 = (
        "..."
)
runner = zlib.decompress(base64.b64decode(RUNNER_B64, validate=True))
if hashlib.sha256(runner).hexdigest() != RUNNER_SHA256:
    raise RuntimeError("Notebook runner checksum mismatch")
temp_root = Path(tempfile.gettempdir())
runner_path = temp_root / f"arc2_runner_{RUNNER_SHA256[:12]}.py"
runner_path.write_bytes(runner)
work_dir = Path("/kaggle/working")
if not work_dir.is_dir():
    work_dir = Path(os.environ.get("ARC2_OUTPUT_DIR", Path.cwd())).resolve()
payload_root = temp_root / f"arc2_payload_{PAYLOAD_SHA256[:12]}"
environment = {**os.environ, "ARC2_PAYLOAD_ROOT": str(payload_root)}
arguments = [sys.executable, str(runner_path), "--output-dir", str(work_dir)]
challenges = os.environ.get("ARC2_CHALLENGES")
if challenges:
    arguments.extend(["--challenges", challenges])
subprocess.run(arguments, check=True, env=environment)
